In [1]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm

from python.config import load_config

cfg = load_config(os.environ.get('CONFIG', 'config.json'))

import python.plots as plots
from python.error_metrics import LoadedErrorMetrics, FilteredErrorMetrics, MetricsIndex as MI # Alias
from python.sampled_hierarchies import SampledHierarchies, SingleStepHierarchies
from python.statistical_test import TrendType, test_stats_matrix, TestResultsLatex, per_series_test, page_trend_test, slope_ttest
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS
from python.aggregation import TOP_LEVEL, BOTTOM_LEVEL, Aggregation

In [2]:
AVAILABLE_LEVELS = Aggregation.complete_summing_matrix(n=cfg.LEVELS.SUBSET, seed=cfg.SEED)

if cfg.is_single_step_hierarchies():
    sh = SingleStepHierarchies(AVAILABLE_LEVELS)
else:
    sh = SampledHierarchies(AVAILABLE_LEVELS, cfg.LEVELS.DIFF, cfg.LEVELS.SEQUENCES, max_h=cfg.LEVELS.MAX_H)

CAPTION_SUFFIX = f"{cfg.DATASET.NAME} dataset - {cfg.MODEL} model"
LABEL_SUFFIX = f"{cfg.DATASET.NAME}_{cfg.MODEL}"

em = FilteredErrorMetrics(
    LoadedErrorMetrics(f'metrics/{cfg.FOLDER}/sample_hiers_errors.csv'),
    MI(error_metric=cfg.ERROR_METRIC)
)

test_fun = page_trend_test if cfg.TEST == 'page l' else slope_ttest

In [ ]:
top_rec_stats_matrix = sh.error_matrix(em, sh.index_matrix(granularity=TOP_LEVEL, forecast_type=RECONCILED_FORECASTS))
bot_rec_stats_matrix = sh.error_matrix(em, sh.index_matrix(granularity=BOTTOM_LEVEL,   forecast_type=RECONCILED_FORECASTS))

base_stats = em.get_error_stats(MI(hierarchy=sh.base_hierarchy, forecast_type=BASE_FORECASTS))

In [ ]:
plots.latex_stats_table(base_stats, cfg.STATISTIC, 'granularity',
                        stat_label='Winsorized Mean NRMSE',
                        caption=fr'Baseline base forecast error statistics (winsorized mean NRMSE at 0.1\%) - {CAPTION_SUFFIX}',
                        label=f'baseline_{LABEL_SUFFIX}')

In [ ]:
plots.latex_sampled_error_table(sh, cfg.STATISTIC, stats_matrix=top_rec_stats_matrix,
                                caption=fr'Winsorized mean NRMSE (0.1\%) at top level granularity per hierarchy - {CAPTION_SUFFIX}',
                                label=f'month_mean_{LABEL_SUFFIX}')

In [ ]:
plots.latex_sampled_error_table(sh, cfg.STATISTIC, stats_matrix=bot_rec_stats_matrix,
                                caption=fr'Winsorized mean NRMSE (0.1\%) at bottom level granularity per hierarchy - {CAPTION_SUFFIX}',
                                label=f'day_mean_{LABEL_SUFFIX}')

In [ ]:
plots.latex_conformism_table(sh,
                             {"Monthly (\\%)": top_rec_stats_matrix,
                              "Daily (\\%)": bot_rec_stats_matrix},
                              cfg.STATISTIC,
                              'decrease',
                              caption=f'Theorem conformism percentage per sequence - {CAPTION_SUFFIX}',
                              label=f'conformism_{LABEL_SUFFIX}')

In [ ]:
TestResultsLatex(
    test_stats_matrix(top_rec_stats_matrix, cfg.STATISTIC, TrendType.DECREASING, test_fun),
).to_table(title=f'Statistical test results for decreasing trends in top level winsorized mean NRMSE - {CAPTION_SUFFIX}',
           star=False,
           label=f'monthly_test_{LABEL_SUFFIX}')

In [ ]:
TestResultsLatex(
    test_stats_matrix(bot_rec_stats_matrix, cfg.STATISTIC, TrendType.DECREASING, test_fun),
).to_table(title=f'Statistical test results for decreasing trends in bottom level winsorized mean NRMSE - {CAPTION_SUFFIX}',
           star=False,
           label=f'monthly_test_{LABEL_SUFFIX}')

In [ ]:
top_level_per_series_test = per_series_test(test_fun, sh, em, forecast_type=RECONCILED_FORECASTS, granularity=TOP_LEVEL, error_metric=cfg.ERROR_METRIC,
                                            trend_type=TrendType.INCREASING)
print(top_level_per_series_test)
plots.plot_per_series_pvalues(top_level_per_series_test, bins=30)

In [ ]:
bottom_level_per_series_test = per_series_test(test_fun, sh, em, forecast_type=RECONCILED_FORECASTS, granularity=BOTTOM_LEVEL, error_metric=cfg.ERROR_METRIC,
                                               trend_type=TrendType.INCREASING)
print(bottom_level_per_series_test)
plots.plot_per_series_pvalues(bottom_level_per_series_test, bins=30)